# Online Retail Prediction System
**Author:** Uddip Bisht  
**Project:** Online Retail Customer Revenue & Segmentation Prediction  
**Dataset:** UCI Online Retail Dataset (~541,909 transactions, Dec 2010 – Dec 2011)  

---

## Project Overview
This notebook implements a complete end-to-end machine learning pipeline on the UCI Online Retail dataset:

1. **Data Download & Loading** — fetch the dataset from UCI ML Repository
2. **Data Cleaning** — remove nulls, cancelled orders, invalid rows
3. **Exploratory Data Analysis (EDA)** — revenue trends, top products, country distribution
4. **Feature Engineering** — RFM (Recency, Frequency, Monetary) + derived features
5. **Model Training**
   - Revenue Prediction: Random Forest Regressor
   - Customer Segmentation: Gradient Boosting Classifier
6. **Model Evaluation** — metrics, confusion matrix, feature importance
7. **Prediction Demo** — interactive inference on custom inputs
8. **Flask API Reference** — endpoint documentation
9. **Streamlit Frontend Code** — full dashboard source


---
## Section 1 — Install Dependencies

In [ ]:
# Install required packages (run once)
import subprocess, sys

packages = [
    "pandas>=1.5.0",
    "numpy>=1.23.0",
    "openpyxl>=3.0.0",
    "scikit-learn>=1.2.0",
    "matplotlib>=3.6.0",
    "seaborn>=0.12.0",
    "plotly>=5.15.0",
    "flask>=2.3.0",
    "flask-cors>=4.0.0",
    "streamlit>=1.28.0",
    "requests>=2.28.0",
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("All dependencies installed.")

---
## Section 2 — Imports

In [ ]:
import os
import sys
import pickle
import warnings
import zipfile
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.ensemble import RandomForestRegressor, GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", "{:.2f}".format)

# Directories
NOTEBOOK_DIR = os.getcwd()
DATA_DIR     = os.path.join(NOTEBOOK_DIR, "data")
MODELS_DIR   = os.path.join(NOTEBOOK_DIR, "models")
os.makedirs(DATA_DIR,   exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

print("Imports complete.")
print(f"Data   dir : {DATA_DIR}")
print(f"Models dir : {MODELS_DIR}")

---
## Section 3 — Data Download

In [ ]:
def download_dataset(data_dir: str) -> str:
    """
    Download the UCI Online Retail dataset if not already present.
    Returns the path to 'Online Retail.xlsx'.
    """
    excel_path = os.path.join(data_dir, "Online Retail.xlsx")
    if os.path.exists(excel_path):
        print(f"[OK] Dataset already exists: {excel_path}")
        return excel_path

    print("[*] Downloading UCI Online Retail dataset (~22 MB) ...")
    zip_path = os.path.join(data_dir, "online_retail.zip")
    url = "https://archive.ics.uci.edu/static/public/352/online+retail.zip"

    urllib.request.urlretrieve(url, zip_path)
    print("[OK] Download complete. Extracting...")

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(data_dir)
    os.remove(zip_path)
    print(f"[OK] Extracted to {data_dir}")
    return excel_path


DATA_PATH = download_dataset(DATA_DIR)

---
## Section 4 — Data Loading & Cleaning

### 4.1  `load_data` — Read Excel / CSV

In [ ]:
def load_data(filepath: str) -> pd.DataFrame:
    """Load the Online Retail dataset from Excel or CSV."""
    if filepath.endswith(".xlsx"):
        df = pd.read_excel(filepath, engine="openpyxl")
    else:
        df = pd.read_csv(filepath, encoding="ISO-8859-1")
    return df


df_raw = load_data(DATA_PATH)
print(f"Raw shape : {df_raw.shape}")
df_raw.head()

In [ ]:
# Dataset info
df_raw.info()

In [ ]:
# Null counts
print("Missing values per column:")
print(df_raw.isnull().sum())

### 4.2  `clean_data` — Remove Invalid Rows

In [ ]:
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean the raw Online Retail DataFrame:
      - Drop rows with missing CustomerID
      - Remove cancelled orders (InvoiceNo starts with 'C')
      - Remove zero / negative Quantity and UnitPrice
      - Parse InvoiceDate to datetime
      - Add TotalPrice = Quantity * UnitPrice
      - Cast CustomerID to int
    """
    df = df.copy()

    before = len(df)
    df.dropna(subset=["CustomerID"], inplace=True)
    print(f"Dropped {before - len(df):,} rows with missing CustomerID")

    before = len(df)
    df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]
    print(f"Dropped {before - len(df):,} cancelled orders")

    before = len(df)
    df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]
    print(f"Dropped {before - len(df):,} rows with non-positive Quantity/UnitPrice")

    df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
    df["TotalPrice"]  = df["Quantity"] * df["UnitPrice"]
    df["CustomerID"]  = df["CustomerID"].astype(int)

    print(f"\nFinal clean shape : {df.shape}")
    return df


df_clean = clean_data(df_raw)
df_clean.head()

---
## Section 5 — Exploratory Data Analysis (EDA)

### 5.1  Key Performance Indicators

In [ ]:
kpis = {
    "Total Transactions": f"{len(df_clean):,}",
    "Unique Customers"  : f"{df_clean['CustomerID'].nunique():,}",
    "Unique Products"   : f"{df_clean['StockCode'].nunique():,}",
    "Countries"         : f"{df_clean['Country'].nunique():,}",
    "Total Revenue"     : f"£{df_clean['TotalPrice'].sum():,.2f}",
    "Avg Order Value"   : f"£{df_clean.groupby('InvoiceNo')['TotalPrice'].sum().mean():,.2f}",
    "Date Range"        : f"{df_clean['InvoiceDate'].min().date()} → {df_clean['InvoiceDate'].max().date()}",
}

for k, v in kpis.items():
    print(f"  {k:<25} {v}")

### 5.2  Monthly Revenue Trend

In [ ]:
def get_monthly_revenue(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["YearMonth"] = df["InvoiceDate"].dt.to_period("M")
    monthly = (
        df.groupby("YearMonth")
        .agg(Revenue=("TotalPrice", "sum"), Orders=("InvoiceNo", "nunique"))
        .reset_index()
    )
    monthly["YearMonth"] = monthly["YearMonth"].astype(str)
    return monthly


df_monthly = get_monthly_revenue(df_clean)

fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()

ax1.fill_between(df_monthly["YearMonth"], df_monthly["Revenue"],
                 alpha=0.4, color="steelblue", label="Revenue (£)")
ax1.plot(df_monthly["YearMonth"], df_monthly["Revenue"],
         color="steelblue", linewidth=2)
ax2.bar(df_monthly["YearMonth"], df_monthly["Orders"],
        alpha=0.3, color="coral", label="Orders")

ax1.set_xlabel("Month")
ax1.set_ylabel("Revenue (£)", color="steelblue")
ax2.set_ylabel("Number of Orders", color="coral")
plt.title("Monthly Revenue & Order Volume", fontsize=14, fontweight="bold")
ax1.tick_params(axis="x", rotation=45)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"£{x/1000:.0f}k"))
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
plt.tight_layout()
plt.show()

df_monthly

### 5.3  Top 15 Products by Revenue

In [ ]:
def get_product_stats(df: pd.DataFrame) -> pd.DataFrame:
    return (
        df.groupby(["StockCode", "Description"])
        .agg(
            TotalQuantitySold=("Quantity", "sum"),
            TotalRevenue=("TotalPrice", "sum"),
            NumOrders=("InvoiceNo", "nunique"),
            AvgUnitPrice=("UnitPrice", "mean"),
        )
        .reset_index()
        .sort_values("TotalRevenue", ascending=False)
    )


df_products = get_product_stats(df_clean)
top15 = df_products.head(15)

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.barh(top15["Description"].str[:40], top15["TotalRevenue"],
               color=plt.cm.Blues_r(np.linspace(0.3, 0.9, len(top15))))
ax.set_xlabel("Total Revenue (£)")
ax.set_title("Top 15 Products by Revenue", fontsize=14, fontweight="bold")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"£{x/1000:.0f}k"))
ax.invert_yaxis()
plt.tight_layout()
plt.show()

top15[["Description", "TotalRevenue", "TotalQuantitySold", "NumOrders"]].head(10)

### 5.4  Revenue by Country (Top 10)

In [ ]:
def get_country_stats(df: pd.DataFrame) -> pd.DataFrame:
    return (
        df.groupby("Country")
        .agg(Revenue=("TotalPrice", "sum"), Customers=("CustomerID", "nunique"))
        .reset_index()
        .sort_values("Revenue", ascending=False)
    )


df_country = get_country_stats(df_clean)
top10_countries = df_country.head(10)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
axes[0].bar(top10_countries["Country"], top10_countries["Revenue"],
            color=sns.color_palette("Blues_r", 10))
axes[0].set_title("Revenue by Country (Top 10)", fontweight="bold")
axes[0].set_ylabel("Revenue (£)")
axes[0].tick_params(axis="x", rotation=45)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"£{x/1e6:.1f}M"))

# Pie chart
axes[1].pie(top10_countries["Revenue"], labels=top10_countries["Country"],
            autopct="%1.1f%%", startangle=140,
            colors=sns.color_palette("pastel", 10))
axes[1].set_title("Revenue Share (Top 10)", fontweight="bold")

plt.tight_layout()
plt.show()

df_country.head(10)

---
## Section 6 — Feature Engineering (RFM)

Build customer-level **Recency · Frequency · Monetary** features plus derived statistics.

In [ ]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build customer-level RFM + engineered features.

    Returns one row per CustomerID with columns:
      Recency, Frequency, Monetary,
      AvgOrderValue, TotalItems, UniqueProducts,
      AvgItemsPerOrder, SegmentLabel
    """
    snapshot_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)

    rfm = df.groupby("CustomerID").agg(
        Recency   =("InvoiceDate", lambda x: (snapshot_date - x.max()).days),
        Frequency =("InvoiceNo",   "nunique"),
        Monetary  =("TotalPrice",  "sum"),
    ).reset_index()

    customer_stats = df.groupby("CustomerID").agg(
        AvgOrderValue  =("TotalPrice", "mean"),
        TotalItems     =("Quantity",   "sum"),
        UniqueProducts =("StockCode",  "nunique"),
        AvgItemsPerOrder=("Quantity",  lambda x: x.sum() / df.loc[x.index, "InvoiceNo"].nunique()),
    ).reset_index()

    features = rfm.merge(customer_stats, on="CustomerID", how="left")

    # Segment label based on Monetary value
    features["SegmentLabel"] = pd.cut(
        features["Monetary"],
        bins=[0, 500, 2000, np.inf],
        labels=["Low Value", "Medium Value", "High Value"],
    )

    return features


features = engineer_features(df_clean)
print(f"Customer feature matrix shape: {features.shape}")
features.head()

In [ ]:
# Descriptive statistics of RFM features
features[["Recency", "Frequency", "Monetary", "AvgOrderValue",
          "TotalItems", "UniqueProducts"]].describe().round(2)

In [ ]:
# Segment distribution
seg_dist = features["SegmentLabel"].value_counts()
print("Customer Segment Distribution:")
print(seg_dist.to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = ["#2563eb", "#f59e0b", "#10b981"]

axes[0].bar(seg_dist.index, seg_dist.values, color=colors)
axes[0].set_title("Customer Count per Segment", fontweight="bold")
axes[0].set_ylabel("Number of Customers")

axes[1].pie(seg_dist.values, labels=seg_dist.index,
            autopct="%1.1f%%", colors=colors, startangle=140)
axes[1].set_title("Segment Share", fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
# RFM scatter plots
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
segment_palette = {"High Value": "#2563eb", "Medium Value": "#f59e0b", "Low Value": "#10b981"}

plots = [
    ("Recency",   "Monetary", "Recency vs Monetary"),
    ("Frequency", "Monetary", "Frequency vs Monetary"),
    ("Recency",   "Frequency", "Recency vs Frequency"),
]

df_plot = features[features["Monetary"] < features["Monetary"].quantile(0.95)]

for ax, (x, y, title) in zip(axes, plots):
    for seg, color in segment_palette.items():
        mask = df_plot["SegmentLabel"] == seg
        ax.scatter(df_plot.loc[mask, x], df_plot.loc[mask, y],
                   c=color, label=seg, alpha=0.5, s=15)
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.set_title(title, fontweight="bold")
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

---
## Section 7 — Model Training

### 7.1  Revenue Prediction — Random Forest Regressor

In [ ]:
REGRESSION_FEATURES     = ["Recency", "Frequency", "AvgOrderValue", "TotalItems", "UniqueProducts"]
CLASSIFICATION_FEATURES = ["Recency", "Frequency", "AvgOrderValue", "TotalItems", "UniqueProducts"]


def train_revenue_model(features_df: pd.DataFrame):
    """
    Train Random Forest Regressor to predict log(Monetary).
    Saves model + scaler to MODELS_DIR.
    Returns: model, scaler, metrics dict
    """
    df = features_df.dropna(subset=REGRESSION_FEATURES + ["Monetary"]).copy()
    X = df[REGRESSION_FEATURES]
    y = np.log1p(df["Monetary"])  # log-transform to reduce skew

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc  = scaler.transform(X_test)

    model = RandomForestRegressor(
        n_estimators=150, max_depth=8, random_state=42, n_jobs=-1
    )
    model.fit(X_train_sc, y_train)

    y_pred = model.predict(X_test_sc)
    metrics = {
        "mae":  round(mean_absolute_error(y_test, y_pred), 4),
        "rmse": round(float(np.sqrt(mean_squared_error(y_test, y_pred))), 4),
        "r2":   round(r2_score(y_test, y_pred), 4),
    }

    with open(os.path.join(MODELS_DIR, "revenue_model.pkl"),  "wb") as f: pickle.dump(model,  f)
    with open(os.path.join(MODELS_DIR, "revenue_scaler.pkl"), "wb") as f: pickle.dump(scaler, f)

    return model, scaler, metrics, (X_test_sc, y_test, y_pred)


rev_model, rev_scaler, rev_metrics, rev_test_data = train_revenue_model(features)

print("Revenue Model (Random Forest Regressor)")
print(f"  MAE  = {rev_metrics['mae']}")
print(f"  RMSE = {rev_metrics['rmse']}")
print(f"  R²   = {rev_metrics['r2']}")

In [ ]:
# Actual vs Predicted scatter (log-space)
_, y_test_rev, y_pred_rev = rev_test_data

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter
axes[0].scatter(y_test_rev, y_pred_rev, alpha=0.4, s=15, color="steelblue")
mn, mx = min(y_test_rev.min(), y_pred_rev.min()), max(y_test_rev.max(), y_pred_rev.max())
axes[0].plot([mn, mx], [mn, mx], "r--", lw=1.5, label="Perfect fit")
axes[0].set_xlabel("Actual log(Revenue)")
axes[0].set_ylabel("Predicted log(Revenue)")
axes[0].set_title(f"Actual vs Predicted  (R² = {rev_metrics['r2']})", fontweight="bold")
axes[0].legend()

# Residuals
residuals = y_test_rev.values - y_pred_rev
axes[1].hist(residuals, bins=40, color="coral", edgecolor="white", alpha=0.8)
axes[1].axvline(0, color="black", lw=1.5, linestyle="--")
axes[1].set_xlabel("Residual (Actual − Predicted)")
axes[1].set_ylabel("Count")
axes[1].set_title("Residual Distribution", fontweight="bold")

plt.tight_layout()
plt.show()

### 7.2  Customer Segmentation — Gradient Boosting Classifier

In [ ]:
def train_segment_model(features_df: pd.DataFrame):
    """
    Train Gradient Boosting Classifier to predict customer segment.
    Saves model, scaler, and label encoder to MODELS_DIR.
    Returns: model, scaler, label_encoder, metrics dict, test data tuple
    """
    df = features_df.dropna(subset=CLASSIFICATION_FEATURES + ["SegmentLabel"]).copy()
    le = LabelEncoder()
    df["SegmentEncoded"] = le.fit_transform(df["SegmentLabel"].astype(str))

    X = df[CLASSIFICATION_FEATURES]
    y = df["SegmentEncoded"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc  = scaler.transform(X_test)

    model = GradientBoostingClassifier(
        n_estimators=100, max_depth=4, random_state=42
    )
    model.fit(X_train_sc, y_train)

    y_pred = model.predict(X_test_sc)
    metrics = {
        "accuracy": round(accuracy_score(y_test, y_pred), 4),
        "report":   classification_report(
            y_test, y_pred, target_names=le.classes_, output_dict=True
        ),
    }

    with open(os.path.join(MODELS_DIR, "segment_model.pkl"),   "wb") as f: pickle.dump(model,  f)
    with open(os.path.join(MODELS_DIR, "segment_scaler.pkl"),  "wb") as f: pickle.dump(scaler, f)
    with open(os.path.join(MODELS_DIR, "segment_encoder.pkl"), "wb") as f: pickle.dump(le,     f)

    return model, scaler, le, metrics, (y_test, y_pred)


seg_model, seg_scaler, seg_le, seg_metrics, seg_test_data = train_segment_model(features)

print("Segmentation Model (Gradient Boosting Classifier)")
print(f"  Accuracy = {seg_metrics['accuracy']}")
print()
print(pd.DataFrame(seg_metrics["report"]).T.round(3))

In [ ]:
# Confusion matrix
y_test_seg, y_pred_seg = seg_test_data
cm = confusion_matrix(y_test_seg, y_pred_seg)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=seg_le.classes_)
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title(f"Confusion Matrix  (Accuracy = {seg_metrics['accuracy']})", fontweight="bold")
plt.tight_layout()
plt.show()

---
## Section 8 — Feature Importance

In [ ]:
def get_feature_importance(model, feature_names: list) -> dict:
    importance = dict(zip(feature_names, model.feature_importances_))
    return {k: round(float(v), 4)
            for k, v in sorted(importance.items(), key=lambda x: -x[1])}


rev_importance = get_feature_importance(rev_model, REGRESSION_FEATURES)
seg_importance = get_feature_importance(seg_model, CLASSIFICATION_FEATURES)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, imp, title in [
    (axes[0], rev_importance, "Revenue Model — Feature Importance"),
    (axes[1], seg_importance, "Segment Model — Feature Importance"),
]:
    keys   = list(imp.keys())
    values = list(imp.values())
    colors = plt.cm.Blues_r(np.linspace(0.3, 0.85, len(keys)))
    ax.barh(keys, values, color=colors)
    ax.set_xlabel("Importance")
    ax.set_title(title, fontweight="bold")
    ax.invert_yaxis()
    for i, v in enumerate(values):
        ax.text(v + 0.002, i, f"{v:.4f}", va="center", fontsize=9)

plt.tight_layout()
plt.show()

print("Revenue model importances:", rev_importance)
print("Segment model importances:", seg_importance)

---
## Section 9 — Prediction Functions & Demo

These are the same inference helpers used by the Flask API.

In [ ]:
def load_revenue_model():
    with open(os.path.join(MODELS_DIR, "revenue_model.pkl"),  "rb") as f: model  = pickle.load(f)
    with open(os.path.join(MODELS_DIR, "revenue_scaler.pkl"), "rb") as f: scaler = pickle.load(f)
    return model, scaler


def load_segment_model():
    with open(os.path.join(MODELS_DIR, "segment_model.pkl"),   "rb") as f: model  = pickle.load(f)
    with open(os.path.join(MODELS_DIR, "segment_scaler.pkl"),  "rb") as f: scaler = pickle.load(f)
    with open(os.path.join(MODELS_DIR, "segment_encoder.pkl"), "rb") as f: le     = pickle.load(f)
    return model, scaler, le


def predict_revenue(recency, frequency, avg_order_value, total_items, unique_products):
    """Predict customer lifetime value (£)."""
    model, scaler = load_revenue_model()
    X = np.array([[recency, frequency, avg_order_value, total_items, unique_products]])
    X_sc = scaler.transform(X)
    log_pred = model.predict(X_sc)[0]
    return round(float(np.expm1(log_pred)), 2)


def predict_segment(recency, frequency, avg_order_value, total_items, unique_products):
    """Predict customer segment label and confidence."""
    model, scaler, le = load_segment_model()
    X = np.array([[recency, frequency, avg_order_value, total_items, unique_products]])
    X_sc = scaler.transform(X)
    encoded_pred = model.predict(X_sc)[0]
    proba        = model.predict_proba(X_sc)[0]
    label        = le.inverse_transform([encoded_pred])[0]
    confidence   = round(float(max(proba)) * 100, 1)
    return label, confidence


print("Prediction functions defined.")

In [ ]:
# ── Prediction Demo ──────────────────────────────────────────────────────────
# Customer profile:
#   Recency          = 30 days
#   Frequency        = 8 orders
#   Avg Order Value  = £180
#   Total Items      = 150
#   Unique Products  = 25

demo_inputs = {
    "recency":         30,
    "frequency":        8,
    "avg_order_value": 180.0,
    "total_items":     150,
    "unique_products":  25,
}

predicted_revenue = predict_revenue(**demo_inputs)
predicted_segment, confidence = predict_segment(**demo_inputs)

print("=" * 45)
print("  Prediction Results")
print("=" * 45)
for k, v in demo_inputs.items():
    print(f"  {k:<22} : {v}")
print("-" * 45)
print(f"  Predicted Revenue    : £{predicted_revenue:,.2f}")
print(f"  Predicted Segment    : {predicted_segment}")
print(f"  Segment Confidence   : {confidence}%")
print("=" * 45)

In [ ]:
# Batch prediction on a sample of existing customers
sample = features[REGRESSION_FEATURES + ["Monetary", "SegmentLabel"]].dropna().head(20).copy()

sample["PredictedRevenue"] = sample.apply(
    lambda r: predict_revenue(
        r.Recency, r.Frequency, r.AvgOrderValue, r.TotalItems, r.UniqueProducts
    ), axis=1
)
sample[["PredLabel", "Confidence"]] = sample.apply(
    lambda r: pd.Series(predict_segment(
        r.Recency, r.Frequency, r.AvgOrderValue, r.TotalItems, r.UniqueProducts
    )), axis=1
)

sample[["Monetary", "PredictedRevenue", "SegmentLabel", "PredLabel", "Confidence"]]

---
## Section 10 — Flask REST API

The cell below contains the complete Flask API source (`backend/app.py`). It is shown here for reference — run it from the terminal with `python backend/app.py`.

In [ ]:
FLASK_APP_CODE = '''
# backend/app.py
# -*- coding: utf-8 -*-
"""
Flask REST API for Online Retail Prediction System.

Endpoints:
  GET  /api/health             - Health check
  GET  /api/summary            - Dataset summary statistics
  GET  /api/monthly-revenue    - Monthly revenue time series
  GET  /api/top-products       - Top N products by revenue
  GET  /api/country-stats      - Revenue by country
  GET  /api/customer-features  - RFM feature table (paginated)
  POST /api/predict/revenue    - Predict customer revenue
  POST /api/predict/segment    - Predict customer segment
  GET  /api/feature-importance - Model feature importances
  POST /api/train              - Retrain all models
"""
import os, sys, traceback
from flask import Flask, jsonify, request
from flask_cors import CORS

sys.path.insert(0, os.path.dirname(__file__))
from data_processor import (load_data, clean_data, engineer_features,
                             get_monthly_revenue, get_product_stats, get_country_stats)
from ml_model import (train_revenue_model, train_segment_model,
                      predict_revenue, predict_segment, get_feature_importance)

app = Flask(__name__)
CORS(app)

DATA_PATH = os.path.join(os.path.dirname(__file__), "..", "data", "Online Retail.xlsx")
_df_clean = _features = None

def get_data():
    global _df_clean, _features
    if _df_clean is None:
        _df_clean  = clean_data(load_data(DATA_PATH))
        _features  = engineer_features(_df_clean)
    return _df_clean, _features

@app.route("/api/health")
def health():
    return jsonify({"status": "ok", "message": "Online Retail API is running"})

@app.route("/api/summary")
def summary():
    df, features = get_data()
    return jsonify({
        "total_transactions": int(len(df)),
        "total_customers":    int(df["CustomerID"].nunique()),
        "total_products":     int(df["StockCode"].nunique()),
        "total_countries":    int(df["Country"].nunique()),
        "total_revenue":      round(float(df["TotalPrice"].sum()), 2),
        "avg_order_value":    round(float(df.groupby("InvoiceNo")["TotalPrice"].sum().mean()), 2),
        "date_range": {
            "start": df["InvoiceDate"].min().strftime("%Y-%m-%d"),
            "end":   df["InvoiceDate"].max().strftime("%Y-%m-%d"),
        },
    })

@app.route("/api/monthly-revenue")
def monthly_revenue():
    df, _ = get_data()
    return jsonify(get_monthly_revenue(df).to_dict(orient="records"))

@app.route("/api/top-products")
def top_products():
    df, _ = get_data()
    n = int(request.args.get("n", 10))
    return jsonify(get_product_stats(df).head(n).to_dict(orient="records"))

@app.route("/api/country-stats")
def country_stats():
    df, _ = get_data()
    return jsonify(get_country_stats(df).to_dict(orient="records"))

@app.route("/api/customer-features")
def customer_features():
    _, features = get_data()
    page, per_page = int(request.args.get("page", 1)), int(request.args.get("per_page", 20))
    start, end = (page - 1) * per_page, page * per_page
    subset = features.iloc[start:end].copy()
    subset["SegmentLabel"] = subset["SegmentLabel"].astype(str)
    return jsonify({"total": len(features), "page": page, "per_page": per_page,
                    "data": subset.to_dict(orient="records")})

@app.route("/api/predict/revenue", methods=["POST"])
def predict_revenue_api():
    data = request.get_json()
    return jsonify({"predicted_revenue": predict_revenue(
        recency=float(data["recency"]), frequency=float(data["frequency"]),
        avg_order_value=float(data["avg_order_value"]),
        total_items=float(data["total_items"]),
        unique_products=float(data["unique_products"]))})

@app.route("/api/predict/segment", methods=["POST"])
def predict_segment_api():
    data = request.get_json()
    label, confidence = predict_segment(
        recency=float(data["recency"]), frequency=float(data["frequency"]),
        avg_order_value=float(data["avg_order_value"]),
        total_items=float(data["total_items"]),
        unique_products=float(data["unique_products"]))
    return jsonify({"segment": label, "confidence": confidence})

@app.route("/api/feature-importance")
def feature_importance_api():
    return jsonify(get_feature_importance())

@app.route("/api/train", methods=["POST"])
def train_models():
    df, features = get_data()
    _, _, rev_metrics = train_revenue_model(features)
    _, _, _, seg_metrics = train_segment_model(features)
    return jsonify({"status": "trained", "revenue_metrics": rev_metrics,
                    "segment_accuracy": seg_metrics["accuracy"]})

if __name__ == "__main__":
    app.run(debug=True, port=5000)
'''

print("Flask API source (backend/app.py) — shown for reference.")
print("Run from terminal:  python backend/app.py")
print()
print(FLASK_APP_CODE[:400], "...")

---
## Section 11 — Streamlit Frontend Source

The full dashboard source lives in `frontend/streamlit_app.py`. Launch it with:
```
streamlit run frontend/streamlit_app.py
```

**Tabs provided:**
| # | Tab | Contents |
|---|-----|----------|
| 1 | 📊 Dashboard | KPI cards, revenue trend, top-country bar chart |
| 2 | 📈 Revenue Trends | Dual-axis monthly revenue + order volume |
| 3 | 🛒 Top Products | Revenue/quantity charts + data table |
| 4 | 🌍 Geography | Choropleth world map + pie chart |
| 5 | 👥 Customers | RFM scatter plots + histograms + paginated table |
| 6 | 🤖 Predict | Interactive prediction form with segment badge + gauge |
| 7 | 📉 Model Insights | Feature importance bar charts + architecture docs |

---
## Section 12 — Project Summary & Results

In [ ]:
print("=" * 60)
print("  ONLINE RETAIL PREDICTION SYSTEM — PROJECT SUMMARY")
print("=" * 60)

print("\n[Dataset]")
print(f"  Raw transactions   : {len(df_raw):,}")
print(f"  After cleaning     : {len(df_clean):,}")
print(f"  Unique customers   : {df_clean['CustomerID'].nunique():,}")
print(f"  Unique products    : {df_clean['StockCode'].nunique():,}")
print(f"  Countries          : {df_clean['Country'].nunique():,}")
print(f"  Total revenue      : £{df_clean['TotalPrice'].sum():,.2f}")

print("\n[Revenue Model — Random Forest Regressor]")
print(f"  MAE                : {rev_metrics['mae']}")
print(f"  RMSE               : {rev_metrics['rmse']}")
print(f"  R²                 : {rev_metrics['r2']}")

print("\n[Segmentation Model — Gradient Boosting Classifier]")
print(f"  Accuracy           : {seg_metrics['accuracy']}")

print("\n[Top Feature Importances (Revenue Model)]")
for feat, imp in list(rev_importance.items())[:3]:
    print(f"  {feat:<22} : {imp}")

print("\n[Segment Distribution]")
for seg, cnt in features["SegmentLabel"].value_counts().items():
    pct = cnt / len(features) * 100
    print(f"  {str(seg):<15} : {cnt:,} customers ({pct:.1f}%)")

print()
print("Models saved to:", MODELS_DIR)
print("=" * 60)